# Treinamento MARL no mundo emergente

Comparamos três configurações no ambiente `emergent`:

- **aleatório**: linha de base sem aprendizado;
- **IQL independente**: uma Q-table por macrófago;
- **IQL compartilhado**: uma única Q-table usada por todos os macrófagos.

A recompensa é de equipe (`ImmuneHomeostasisReward`): redução de patógeno,
menos inflamação e dano, mais reparo.

In [ ]:
import random

import numpy as np

from immune_rl.dynamics import ImmuneHomeostasisReward
from immune_rl.emergent import EmergentDynamics, build_seed_graph
from immune_rl.encoding import EmergentStateEncoder
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import QLearningPolicy, RandomPolicy

In [ ]:
EPISODES = 300
EVAL_EPISODES = 50
SEEDS = 3
MAX_STEPS = 60

In [ ]:
def make_env(seed=0):
    return ImmuneEnvironment(
        build_seed_graph,
        EmergentDynamics(seed=seed),
        ImmuneHomeostasisReward(),
        max_steps=MAX_STEPS,
    )


def ensure_policy(policies, cell_id, kind, encoder, seed):
    if cell_id in policies:
        return policies[cell_id]
    if kind == "shared":
        shared = next(
            (p for p in policies.values() if isinstance(p, QLearningPolicy)), None
        )
        if shared is None:
            shared = QLearningPolicy(encoder, seed=seed)
        policies[cell_id] = shared
    elif kind == "independent":
        policies[cell_id] = QLearningPolicy(encoder, seed=seed + len(policies))
    else:
        policies[cell_id] = RandomPolicy()
    return policies[cell_id]


def run_episode(env, policies, kind, encoder, seed, train=True):
    obs = env.reset()
    total = 0.0
    while True:
        for cell_id in obs:
            ensure_policy(policies, cell_id, kind, encoder, seed)
        actions = {
            cell_id: policies[cell_id].select_action(observation)
            for cell_id, observation in obs.items()
        }
        next_obs, rewards, terminated, truncated, _ = env.step(actions)
        if train:
            for cell_id in obs:
                if cell_id in next_obs and cell_id in rewards:
                    policies[cell_id].update(
                        obs[cell_id],
                        actions[cell_id],
                        rewards[cell_id],
                        next_obs[cell_id],
                        terminated,
                    )
        total += sum(rewards.values()) / max(len(rewards), 1)
        obs = next_obs
        if terminated or truncated:
            if train:
                for policy in set(policies.values()):
                    policy.end_episode()
            return total


def experiment(kind, seed):
    random.seed(seed)
    env = make_env(seed)
    encoder = EmergentStateEncoder()
    policies = {}
    train_returns = [
        run_episode(env, policies, kind, encoder, seed, train=True)
        for _ in range(EPISODES)
    ]
    for policy in set(policies.values()):
        if isinstance(policy, QLearningPolicy):
            policy.epsilon = 0.0
    eval_returns = [
        run_episode(env, policies, kind, encoder, seed, train=False)
        for _ in range(EVAL_EPISODES)
    ]
    return (
        float(np.mean(train_returns[:50])),
        float(np.mean(train_returns[-50:])),
        float(np.mean(eval_returns)),
        len(policies),
    )

In [ ]:
configs = [
    ("random", "aleatório"),
    ("independent", "IQL independente"),
    ("shared", "IQL compartilhado"),
]

print(f"{'configuração':<20}{'início':>9}{'fim treino':>12}{'avaliação':>11}")
results = {}
for kind, name in configs:
    runs = np.array([experiment(kind, seed)[:3] for seed in range(SEEDS)])
    mean = runs.mean(axis=0)
    results[kind] = mean
    print(f"{name:<20}{mean[0]:>9.3f}{mean[1]:>12.3f}{mean[2]:>11.3f}")

## Leitura dos resultados

O IQL compartilhado tende a aprender mais rápido porque todos os macrófagos
alimentam a mesma Q-table. O IQL independente tem mais capacidade, mas sofre
com a não-estacionariedade dos outros agentes. A linha de base aleatória mostra
o quanto a recompensa vem da dinâmica, não do aprendizado.

Para inspecionar a Q-table aprendida e uma trajetória narrada, abra
`confere_aprendizado.ipynb`.